# Customer Support Routing Agent

## Use Case

Handling customer support requests manually is time-consuming and inconsistent. 
This notebook implements an agentic routing system that automatically classifies 
incoming support requests by **topic** and **sentiment**, then routes them to the 
right specialized agent or escalates to a human when needed.

This pattern is directly applicable to any support, helpdesk, or triage system 
where volume and variety make manual routing impractical.

## Architecture
```mermaid
graph TD
    A[User Input] --> B[classify_topic]
    A --> C[classify_sentiment]
    B --> D{router}
    C --> D
    D -->|billing| E[billing_agent]
    D -->|tech + neutral/positive| F[tech_agent]
    D -->|account| G[account_agent]
    D -->|general| H[general_agent]
    D -->|other / angry| I[escalate]
    E --> J[Output]
    F --> J
    G --> J
    H --> J
    I --> J
```

**Routing logic:**
- `angry` sentiment → escalate (overrides topic)
- `tech` + `frustrated` → escalate
- `other` → escalate
- All other combinations → specialized agent

## Stack

- **Runtime**: Deno + TypeScript
- **LLM**: OpenAI `gpt-4o-mini`
- **Framework**: LangGraph
- **Libs**: `zod`, `@langchain/openai`, `@std/dotenv`

## What's next?

This notebook uses naive agent nodes (simple prompts). Here are some natural extensions:

- **Specialized sub-graphs** — replace each agent node with a full LangGraph 
  subgraph (e.g. tech_agent with its own diagnosis → solution → validation loop)
- **Human-in-the-loop** — add a checkpointer and interrupt the graph before 
  escalation for human review
- **MCP integration** — connect the escalate node to a mail or ticketing tool 
  via MCP (e.g. send escalation email, create Jira ticket)
- **Memory** — inject customer history into agent prompts for personalized responses
- **Observability** — plug LangSmith or Langfuse to trace every routing decision

## Dependencies

In [ ]:
%pip install langchain langchain-openai langgraph pydantic python-dotenv

## Setup

In [ ]:
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langgraph.graph import StateGraph, END
from pydantic import BaseModel
from typing import Literal
import asyncio

load_dotenv()

llm = ChatOpenAI(model="gpt-5-mini")

## Agent state

In [ ]:
from typing import Literal

Topic =Literal["billing", "tech", "account", "general", "other"]
Sentiment = Literal["positive", "neutral", "frustrated", "angry"]

class SupportState(BaseModel):
    input: str
    topic: Topic = "other"
    sentiment: Sentiment = "neutral"
    response: str = ""
    escalated: bool = False

## Structured output schemas

In [ ]:
class TopicClassification(BaseModel):
    topic: Literal["billing", "tech", "account", "general", "other"]
    reason: str

class SentimentClassification(BaseModel):
    sentiment: Literal["positive", "neutral", "frustrated", "angry"]
    reason: str

## Nodes - Classification

In [ ]:
def classify_topic(state: SupportState) -> dict:
    prompt = f"""You are a customer support classifier.
    
Classify the following support request into one of these categories:
- billing: payment, invoice, subscription, refund
- tech: bug, error, performance, integration
- account: access, permissions, login, profile
- general: general question, how-to, documentation
- other: anything that doesn't fit the above

Support request: {state.input}"""

    structured_llm = llm.with_structured_output(TopicClassification)
    result = structured_llm.invoke(prompt)
    topic = TopicClassification.model_validate(result)
    return {"topic": topic}


def classify_sentiment(state: SupportState) -> dict:
    prompt = f"""You are a sentiment analyzer specialized in customer support.

Classify the sentiment of the following support request:
- positive: satisfied, polite, calm
- neutral: factual, no strong emotion
- frustrated: clearly annoyed but still constructive
- angry: aggressive, threatening, highly emotional

Support request: {state.input}"""

    structured_llm = llm.with_structured_output(SentimentClassification)
    result = structured_llm.invoke(prompt)
    sentiment = SentimentClassification.model_validate(result)
    return {"sentiment": sentiment}

## Nodes - Router

In [ ]:
def router(state: SupportState) -> str:
    # Angry overrides everything
    if state.sentiment == "angry":
        return "escalate"
    
    # Tech + frustrated → escalate
    if state.topic == "tech" and state.sentiment == "frustrated":
        return "escalate"
    
    # Other → escalate
    if state.topic == "other":
        return "escalate"
    
    # Route to specialized agent
    return f"{state.topic}_agent" 

## Nodes - Agents

In [ ]:
def billing_agent(state: SupportState) -> dict:
    prompt = f"""You are a billing support specialist.
    
Provide a helpful, professional response to this billing request.
Be concise and solution-oriented.

Request: {state.input}"""

    response = llm.invoke(prompt)
    return {"response": response.content}


def tech_agent(state: SupportState) -> dict:
    prompt = f"""You are a technical support specialist.
    
Provide a helpful, professional response to this technical request.
Ask clarifying questions if needed. Be precise and structured.

Request: {state.input}"""

    response = llm.invoke(prompt)
    return {"response": response.content}


def account_agent(state: SupportState) -> dict:
    prompt = f"""You are an account management specialist.
    
Provide a helpful, professional response to this account request.
Be reassuring and clear about next steps.

Request: {state.input}"""

    response = llm.invoke(prompt)
    return {"response": response.content}


def general_agent(state: SupportState) -> dict:
    prompt = f"""You are a customer support generalist.
    
Provide a helpful, professional response to this request.
Be friendly and thorough.

Request: {state.input}"""

    response = llm.invoke(prompt)
    return {"response": response.content}


def escalate(state: SupportState) -> dict:
    prompt = f"""You are a support escalation specialist.

A support request requires human attention. Write a concise escalation summary 
including: the customer's issue, detected sentiment, and recommended priority level.

Request: {state.input}
Detected topic: {state.topic}
Detected sentiment: {state.sentiment}"""

    response = llm.invoke(prompt)
    return {"response": response.content, "escalated": True}

## Build graph

In [ ]:
from langgraph.graph import StateGraph, END

builder = StateGraph(SupportState)

# Add nodes
builder.add_node("classify_topic", classify_topic)
builder.add_node("classify_sentiment", classify_sentiment)
builder.add_node("router", lambda state: state)  # passthrough, routing via conditional edges
builder.add_node("billing_agent", billing_agent)
builder.add_node("tech_agent", tech_agent)
builder.add_node("account_agent", account_agent)
builder.add_node("general_agent", general_agent)
builder.add_node("escalate", escalate)

# Parallel classification
builder.add_edge("__start__", "classify_topic")
builder.add_edge("__start__", "classify_sentiment")


builder.add_edge("classify_sentiment", "router")
builder.add_edge("classify_topic", "router")

# All agents to END
builder.add_edge("billing_agent", END)
builder.add_edge("tech_agent", END)
builder.add_edge("account_agent", END)
builder.add_edge("general_agent", END)
builder.add_edge("escalate", END)

# Conditional routing
builder.add_conditional_edges("router", router, {
    "billing_agent": "billing_agent",
    "tech_agent": "tech_agent",
    "account_agent": "account_agent",
    "general_agent": "general_agent",
    "escalate": "escalate"
})

graph = builder.compile()

## Visualize graph

In [ ]:
from IPython.display import Image, display

display(Image(graph.get_graph().draw_mermaid_png()))

## Some tests

In [ ]:
test_cases = [
    "I was charged twice this month, I need a refund immediately.",
    "Hi, I can't seem to login to my account since this morning.",
    "Your app is completely broken and I've lost 3 hours of work because of you, this is unacceptable!!!",
    "How do I export my data to CSV?",
    "The API is returning 500 errors intermittently on the /users endpoint.",
    "I'd like to update my billing email address.",
]

for input_text in test_cases:
    result = graph.invoke(SupportState(input=input_text))
    print(f"📩 Input: {input_text[:60]}...")
    print(f"🏷️  Topic: {result['topic']} | 😐 Sentiment: {result['sentiment']}")
    print(f"{'🚨 ESCALATED' if result['escalated'] else '✅ Handled'}")
    print(f"💬 Response: {result['response'][:120]}...")
    print("-" * 80)